# NBS CRM User Clustering

Monthly re-run: extract → preprocess → cluster → profile → export.
Spec: `docs/superpowers/specs/2026-05-18-crm-user-clustering-design.md`

In [ ]:
import os
import warnings
from datetime import datetime

import hdbscan
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()

engine = create_engine(os.environ["READONLY_DATABASE_URL"], pool_pre_ping=True)
ANALYSIS_MONTH = datetime.now().strftime("%Y-%m")
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Analysis month: {ANALYSIS_MONTH}")
print("DB engine ready:", engine.url.host)

In [ ]:
# ── Users base ──────────────────────────────────────────────────────────────
sql_users = text("""
    SELECT
        u.id                                    AS user_id,
        u.kyc_level,
        u.status,
        u.created_at,
        u.last_active_at,
        (u.account_type = 'business')::int      AS account_type_business,
        COALESCE(p.onboarding_completed, FALSE)::int AS onboarding_completed
    FROM users u
    LEFT JOIN user_profiles p ON p.user_id = u.id
    WHERE u.status != 'suspended'
""")

df_users = pd.read_sql(sql_users, engine)
print(f"Users: {len(df_users):,}")
assert df_users["user_id"].nunique() == len(df_users), "Duplicate user_ids"
assert df_users["kyc_level"].between(0, 3).all(), "kyc_level out of range"
df_users.head(3)

In [ ]:
# ── Founders ─────────────────────────────────────────────────────────────────
sql_founders = text("""
    SELECT
        user_id,
        1                   AS is_founder,
        network_size        AS founder_network_size,
        invites_sent
    FROM founders
""")

df_founders = pd.read_sql(sql_founders, engine)
assert df_founders["user_id"].nunique() == len(df_founders), "Duplicate user_ids in founders"
assert (df_founders["founder_network_size"] >= 0).all(), "Negative network_size found"
assert (df_founders["invites_sent"] >= 0).all(), "Negative invites_sent found"
print(f"Founders: {len(df_founders):,}")
df_founders.head(3)

In [ ]:
# ── Onramp / Offramp ─────────────────────────────────────────────────────────
# from_amount_brl is NULL on offramp rows; to_amount_brl is NULL on onramp rows.
# COALESCE(..., 0) before every SUM.
sql_conversions = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (WHERE direction = 'brl_to_usdc')    AS n_onramp_txns,
        COUNT(*) FILTER (WHERE direction = 'usdc_to_brl')    AS n_offramp_txns,
        COALESCE(SUM(
            CASE WHEN direction = 'brl_to_usdc'
                 THEN COALESCE(from_amount_brl, 0) ELSE 0 END
        ) / 100.0, 0)                                        AS total_onramp_brl,
        COALESCE(SUM(
            COALESCE(spread_revenue_brl, 0) + COALESCE(fee_amount_brl, 0)
        ) / 100.0, 0)                                        AS total_spread_revenue_brl,
        COALESCE(
            SUM((processing_mode = 'instant')::int)::float
            / NULLIF(COUNT(*), 0), 0
        )                                                    AS pct_instant_mode,
        MAX(created_at)                                      AS last_conversion_at
    FROM conversion_quotes
    WHERE used = TRUE
    GROUP BY user_id
""")

df_conversions = pd.read_sql(sql_conversions, engine)
print(f"Users with conversions: {len(df_conversions):,}")
assert (df_conversions["total_spread_revenue_brl"] >= 0).all(), "Negative revenue"
assert df_conversions["user_id"].nunique() == len(df_conversions), "Duplicate user_ids"
df_conversions.head(3)

In [ ]:
# ── Cards ────────────────────────────────────────────────────────────────────
sql_cards_txns = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (
            WHERE transaction_type = 'spend' AND status = 'completed'
        )                                   AS n_card_txns,
        COALESCE(SUM(amount) FILTER (
            WHERE transaction_type = 'spend' AND status = 'completed'
        ) / 100.0, 0)                       AS total_card_spend_usd,
        MAX(authorized_at) FILTER (
            WHERE transaction_type = 'spend'
        )                                   AS last_card_spend_at
    FROM card_transactions
    GROUP BY user_id
""")

sql_cards_issued = text("""
    SELECT
        user_id,
        1                                   AS has_card,
        MAX((card_variant = 'founder')::int) AS card_variant_founder
    FROM cards
    WHERE status = 'active'
    GROUP BY user_id
""")

sql_annual_fee = text("""
    SELECT DISTINCT user_id, 1 AS paid_annual_fee
    FROM card_annual_fees
    WHERE status = 'paid'
""")

df_card_txns   = pd.read_sql(sql_cards_txns, engine)
df_card_issued = pd.read_sql(sql_cards_issued, engine)
df_annual_fee  = pd.read_sql(sql_annual_fee, engine)

assert df_annual_fee["user_id"].nunique() == len(df_annual_fee), "Duplicate user_ids in annual_fee"

assert (df_card_txns["total_card_spend_usd"] >= 0).all(), "Negative card spend"
assert df_card_txns["user_id"].nunique() == len(df_card_txns), "Duplicate user_ids in card_txns"
assert df_card_issued["user_id"].nunique() == len(df_card_issued), "Duplicate user_ids in card_issued"

print(f"Users with card txns:   {len(df_card_txns):,}")
print(f"Users with active card: {len(df_card_issued):,}")
print(f"Users paid annual fee:  {len(df_annual_fee):,}")

In [ ]:
# ── Swaps + Solana ────────────────────────────────────────────────────────────
sql_swaps = text("""
    SELECT
        user_id,
        COUNT(*)                                            AS n_swaps,
        COALESCE(SUM(input_amount) / 1e6, 0)               AS total_swap_volume_usdc,
        COUNT(DISTINCT input_mint)
            + COUNT(DISTINCT output_mint)                   AS n_unique_tokens,
        MAX(timestamp)                                      AS last_swap_at
    FROM swap_transactions
    GROUP BY user_id
""")

sql_solana = text("""
    SELECT
        user_id,
        COUNT(DISTINCT transaction_signature) AS n_solana_txns
    FROM solana_sponsored_transactions
    GROUP BY user_id
""")

df_swaps  = pd.read_sql(sql_swaps, engine)
df_solana = pd.read_sql(sql_solana, engine)

assert df_swaps["user_id"].nunique() == len(df_swaps), "Duplicate user_ids in swaps"
assert (df_swaps["total_swap_volume_usdc"] >= 0).all(), "Negative swap volume"
assert df_solana["user_id"].nunique() == len(df_solana), "Duplicate user_ids in solana"

print(f"Users with swaps:  {len(df_swaps):,}")
print(f"Users with Solana: {len(df_solana):,}")

In [ ]:
# ── AI sessions ──────────────────────────────────────────────────────────────
sql_ai = text("""
    SELECT
        user_id,
        COUNT(*)                        AS n_ai_sessions,
        COALESCE(SUM(message_count), 0) AS total_ai_messages
    FROM ai_sessions
    GROUP BY user_id
""")

# ── Notification engagement ───────────────────────────────────────────────
sql_notifications = text("""
    SELECT
        user_id,
        COUNT(*)                                        AS n_notifications_received,
        COUNT(*) FILTER (WHERE read_at IS NOT NULL)     AS n_notifications_read
    FROM notification_events
    GROUP BY user_id
""")

# ── International payouts ─────────────────────────────────────────────────
sql_international = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (WHERE status = 'completed')        AS n_international_payouts,
        COALESCE(SUM(amount) FILTER (WHERE status = 'completed'), 0)
                                                            AS total_international_usdc
    FROM unblockpay_payouts
    GROUP BY user_id
""")

df_ai            = pd.read_sql(sql_ai, engine)
df_notifications = pd.read_sql(sql_notifications, engine)
df_international = pd.read_sql(sql_international, engine)

assert df_ai["user_id"].nunique() == len(df_ai), "Duplicate user_ids in ai"
assert df_notifications["user_id"].nunique() == len(df_notifications), "Duplicate user_ids in notifications"
assert (df_notifications["n_notifications_read"] <= df_notifications["n_notifications_received"]).all(), \
    "Read count exceeds received count"

print(f"Users with AI sessions:   {len(df_ai):,}")
print(f"Users with notifications: {len(df_notifications):,}")
print(f"Users with intl payouts:  {len(df_international):,}")

In [ ]:
# ── Merge all sources onto user base (left joins — zero-fill = genuine non-usage) ─
df = df_users.copy()

for tbl in [
    df_founders,
    df_conversions,
    df_card_txns,
    df_card_issued,
    df_annual_fee,
    df_swaps,
    df_solana,
    df_ai,
    df_notifications,
    df_international,
]:
    df = df.merge(tbl, on="user_id", how="left")

print(f"Shape after merge: {df.shape}")
assert len(df) == len(df_users), "Row count changed after merges — check for duplicates"


In [ ]:
# ── Recency features ─────────────────────────────────────────────────────────
# DB datetimes are tz-aware UTC — strip timezone before .dt.days arithmetic
now = pd.Timestamp.now().normalize()

def recency_days(col):
    return (now - pd.to_datetime(df[col]).dt.tz_convert(None)).dt.days.fillna(9999).clip(upper=730)

df["days_since_signup"]          = recency_days("created_at")
df["days_since_last_active"]     = recency_days("last_active_at")
df["days_since_last_conversion"] = recency_days("last_conversion_at")
df["days_since_last_card_spend"] = recency_days("last_card_spend_at")
df["days_since_last_swap"]       = recency_days("last_swap_at")

# Zero-fill all numeric NaN (non-usage, not missing data)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols] = df[numeric_cols].fillna(0)

print("NaN remaining:", df[numeric_cols].isna().sum().sum())
assert df[numeric_cols].isna().sum().sum() == 0, "Unexpected NaN in numeric columns"


In [ ]:
# ── Binary presence flags ────────────────────────────────────────────────────
df["has_onramp"]  = (df["n_onramp_txns"] > 0).astype(int)
df["has_offramp"] = (df["n_offramp_txns"] > 0).astype(int)
df["has_card"]    = df["has_card"].fillna(0).astype(int)
df["has_swap"]    = (df["n_swaps"] > 0).astype(int)
df["has_ai"]      = (df["n_ai_sessions"] > 0).astype(int)
df["has_intl"]    = (df["n_international_payouts"] > 0).astype(int)

# ── Product breadth score (0–6) ──────────────────────────────────────────────
df["product_breadth_score"] = (
    df["has_onramp"] + df["has_offramp"] + df["has_card"]
    + df["has_swap"] + df["has_ai"] + df["has_intl"]
)

# ── Revenue generated (spread only) ──────────────────────────────────────────
df["revenue_generated_brl"] = df["total_spread_revenue_brl"]

# ── Notification read rate ────────────────────────────────────────────────────
df["notification_read_rate"] = (
    df["n_notifications_read"]
    / df["n_notifications_received"].replace(0, np.nan)
).fillna(0)

print("Breadth score distribution:")
print(df["product_breadth_score"].value_counts().sort_index())


In [ ]:
# ── Feature column lists ─────────────────────────────────────────────────────
LOG_FEATURES = [
    "n_onramp_txns", "n_offramp_txns", "total_onramp_brl",
    "total_spread_revenue_brl", "n_card_txns", "total_card_spend_usd",
    "n_swaps", "total_swap_volume_usdc", "n_unique_tokens",
    "n_solana_txns", "n_ai_sessions", "total_ai_messages",
    "n_notifications_received", "n_international_payouts",
    "total_international_usdc", "founder_network_size", "invites_sent",
    "revenue_generated_brl",
]

BINARY_FLAGS = [
    "has_onramp", "has_offramp", "has_card", "has_swap", "has_ai", "has_intl",
    "is_founder", "paid_annual_fee", "card_variant_founder",
    "onboarding_completed", "account_type_business",
]

NUMERIC_FEATURES = [
    "days_since_signup", "days_since_last_active", "days_since_last_conversion",
    "days_since_last_card_spend", "days_since_last_swap",
    "kyc_level", "product_breadth_score", "notification_read_rate",
    "pct_instant_mode",
]

ALL_FEATURES = LOG_FEATURES + BINARY_FLAGS + NUMERIC_FEATURES

X_raw = df[ALL_FEATURES].copy()

# log1p on skewed count/monetary features — must be non-negative (assert first)
assert (X_raw[LOG_FEATURES] >= 0).all().all(), "Negative values in log features"
X_raw[LOG_FEATURES] = np.log1p(X_raw[LOG_FEATURES])

print(f"Feature matrix shape: {X_raw.shape}")

In [ ]:
# ── Scale ────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ── PCA (retain 90% variance) ─────────────────────────────────────────────────
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_scaled)

n_components = X_pca.shape[1]
variance_explained = pca.explained_variance_ratio_.sum()
print(f"PCA: {n_components} components → {variance_explained:.1%} variance retained")
assert 6 <= n_components <= 16, f"Unexpected component count: {n_components}"

# Variance explained per component (sanity check)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, n_components + 1), pca.explained_variance_ratio_ * 100)
ax.set(xlabel="Component", ylabel="Variance explained (%)",
       title="PCA — Variance per component")
plt.tight_layout()
plt.savefig(f"{PROCESSED_DIR}/crm_pca_variance_{ANALYSIS_MONTH}.png", dpi=120)
plt.show()

In [ ]:
# ── Outlier quarantine ────────────────────────────────────────────────────────
detector = hdbscan.HDBSCAN(
    min_cluster_size=15, min_samples=5, metric="euclidean"
)
outlier_labels = detector.fit_predict(X_pca)
outlier_mask   = outlier_labels == -1

print(f"Outliers identified: {outlier_mask.sum()} ({outlier_mask.mean():.1%} of users)")

In [ ]:
# ── Inspect outliers ──────────────────────────────────────────────────────────
# Are they bots, internal accounts, or data anomalies?
df_outliers = df[outlier_mask][[
    "user_id", "n_onramp_txns", "total_card_spend_usd",
    "n_swaps", "n_solana_txns", "days_since_last_active",
    "kyc_level", "product_breadth_score",
]].copy()

print(df_outliers.describe())
print(f"\nOutlier KYC distribution:\n{df_outliers['kyc_level'].value_counts()}")
print(f"\nOutlier breadth distribution:\n{df_outliers['product_breadth_score'].value_counts()}")

In [ ]:
# ── Remove outliers before final clustering ───────────────────────────────────
X_pca_clean = X_pca[~outlier_mask]
X_raw_clean = X_raw[~outlier_mask].reset_index(drop=True)
df_clean    = df[~outlier_mask].copy().reset_index(drop=True)

print(f"Clean population: {len(df_clean):,} users ({outlier_mask.sum()} quarantined)")
assert len(df_clean) > 8_000, "Too many users quarantined — check HDBSCAN params"

In [ ]:
# ── K selection: silhouette + Davies-Bouldin + inertia ───────────────────────
k_results = []

for k in range(4, 12):
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42)
    labels = km.fit_predict(X_pca_clean)
    sil = silhouette_score(X_pca_clean, labels, sample_size=5_000, random_state=42)
    db  = davies_bouldin_score(X_pca_clean, labels)
    k_results.append({
        "k": k,
        "silhouette": sil,
        "davies_bouldin": db,
        "inertia": km.inertia_,
    })
    print(f"k={k}: silhouette={sil:.3f}  DB={db:.3f}  inertia={km.inertia_:.0f}")

results_df = pd.DataFrame(k_results)

In [ ]:
# ── Diagnostic plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(results_df["k"], results_df["silhouette"], "o-", color="steelblue")
axes[0].set(title="Silhouette (higher = better)", xlabel="k", ylabel="Score")

axes[1].plot(results_df["k"], results_df["davies_bouldin"], "o-", color="darkorange")
axes[1].set(title="Davies-Bouldin (lower = better)", xlabel="k", ylabel="Score")

axes[2].plot(results_df["k"], results_df["inertia"], "o-", color="green")
axes[2].set(title="Inertia — elbow curve", xlabel="k", ylabel="Inertia")

plt.suptitle("K Selection Diagnostics", y=1.02)
plt.tight_layout()
plt.savefig(f"{PROCESSED_DIR}/crm_k_selection_{ANALYSIS_MONTH}.png", dpi=120)
plt.show()

best_k = int(results_df.loc[results_df["silhouette"].idxmax(), "k"])
print(f"\nBest k by silhouette: {best_k}")
print("Review the elbow plot and confirm — override BEST_K below if needed.")

In [ ]:
# ── Set BEST_K here after reviewing the diagnostic plots ─────────────────────
# Default: silhouette peak. Override if elbow or business narrative suggests otherwise.
BEST_K = best_k   # change this integer if needed
print(f"Using BEST_K = {BEST_K}")